# TP IA 2025 - Entrenamiento Final
## Parte 2: Entrenamiento con Mejor Modelo y Evaluación

Este notebook:
1. Carga la configuración óptima del AG
2. Prepara datos (predicción de alcohol_freq incluida)
3. Entrena el modelo final con 70% de datos
4. Evalúa con 30% restante

**Tiempo estimado**: ~1 hora

**Prerequisito**: Ejecutar primero `19-IA2025-TP-Preparacion-Optimizacion.ipynb`

In [ ]:
# Importaciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers import Adam

print(f"TensorFlow: {tf.__version__}")
print(f"Inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

np.random.seed(42)
tf.random.set_seed(42)

sns.set(style='whitegrid', palette='muted', font_scale=1.2)

start_total = time.time()

## 1. Cargar Configuración Óptima

In [ ]:
# Cargar configuración del mejor modelo
with open('best_model_config.json', 'r') as f:
    best_config = json.load(f)

print("="*80)
print("CONFIGURACIÓN ÓPTIMA CARGADA")
print("="*80)
print(f"\nArquitectura:")
print(f"  Capas: {best_config['n_layers']}")
print(f"  Nodos por capa: {best_config['nodes_per_layer']}")
print(f"  Batch Normalization: {best_config['use_batch_norm']}")
if best_config['use_batch_norm']:
    print(f"    - Momentum: {best_config['batch_norm_momentum']:.4f}")
print(f"  Dropout rates: {[f'{d:.3f}' for d in best_config['dropout_rates']]}")
print(f"  Activación: {best_config['activation']}")
print(f"  Learning Rate: {best_config['learning_rate']:.6f}")
print(f"\nRendimiento en validación AG:")
print(f"  MAE: {best_config['mae_validation']:.4f}")
print(f"  Mejora sobre baseline: {best_config['improvement_percentage']:.2f}%")
print("="*80)

## 2. Carga y Preparación de Datos

In [ ]:
# Cargar dataset
dirr = "/home/pedro_dev/Pedro/IA/competencia_FF/19-IA2025 medical_insurance.csv"
df = pd.read_csv(dirr)

print(f"\nDataset: {df.shape[0]} registros, {df.shape[1]} columnas")

# Convertir categóricas
categorical_cols = ['sex', 'region', 'urban_rural', 'education', 'marital_status', 
                   'employment_status', 'smoker', 'alcohol_freq', 'plan_type', 'network_tier']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

## 3. Predicción Rápida de alcohol_freq
(Usando arquitectura optimizada del notebook de preparación)

In [ ]:
print("\n" + "="*80)
print("PREDICCIÓN DE ALCOHOL_FREQ")
print("="*80)

# Verificar si hay valores faltantes
n_missing = df['alcohol_freq'].isna().sum()
print(f"\nValores faltantes: {n_missing}")

if n_missing > 0:
    # Separar datos
    df_valid = df[df['alcohol_freq'].notna()].copy()
    df_missing = df[df['alcohol_freq'].isna()].copy()
    
    # Codificar
    label_encoder = LabelEncoder()
    y_alcohol = label_encoder.fit_transform(df_valid['alcohol_freq'])
    n_classes = len(label_encoder.classes_)
    
    # Preparar features (INCLUYE annual_medical_cost)
    exclude_for_alcohol = ['person_id', 'alcohol_freq']
    df_valid_features = df_valid.drop(columns=exclude_for_alcohol, errors='ignore')
    
    categorical_cols_pred = ['sex', 'region', 'urban_rural', 'education', 'marital_status', 
                             'employment_status', 'smoker', 'plan_type', 'network_tier']
    
    X_alcohol = pd.get_dummies(df_valid_features, 
                               columns=[c for c in categorical_cols_pred if c in df_valid_features.columns], 
                               drop_first=True)
    X_alcohol = X_alcohol.fillna(X_alcohol.median())
    
    # Split y normalizar
    from tensorflow.keras.utils import to_categorical
    from sklearn.preprocessing import StandardScaler
    from tensorflow.keras import regularizers
    
    X_alc_train, X_alc_test, y_alc_train, y_alc_test = train_test_split(
        X_alcohol, y_alcohol, test_size=0.20, random_state=42, stratify=y_alcohol
    )
    
    scaler_alcohol = StandardScaler()
    X_alc_train_scaled = scaler_alcohol.fit_transform(X_alc_train)
    X_alc_test_scaled = scaler_alcohol.transform(X_alc_test)
    
    y_alc_train_cat = to_categorical(y_alc_train, num_classes=n_classes)
    y_alc_test_cat = to_categorical(y_alc_test, num_classes=n_classes)
    
    # Modelo con dropout
    model_alcohol = models.Sequential([
        layers.Input(shape=(X_alc_train_scaled.shape[1],)),
        layers.Dense(1024, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.3),
        layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.2),
        layers.Dense(n_classes, activation='softmax')
    ])
    
    model_alcohol.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    print("\nEntrenando modelo de alcohol_freq...")
    
    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=8, restore_best_weights=True, verbose=0
    )
    
    history_alcohol = model_alcohol.fit(
        X_alc_train_scaled, y_alc_train_cat,
        validation_data=(X_alc_test_scaled, y_alc_test_cat),
        epochs=100,
        batch_size=128,
        callbacks=[early_stop],
        verbose=0
    )
    
    # Predecir faltantes
    df_missing_features = df_missing.drop(columns=exclude_for_alcohol, errors='ignore')
    X_missing = pd.get_dummies(df_missing_features, 
                              columns=[c for c in categorical_cols_pred if c in df_missing_features.columns],
                              drop_first=True)
    
    for col in X_alcohol.columns:
        if col not in X_missing.columns:
            X_missing[col] = 0
    X_missing = X_missing[X_alcohol.columns]
    X_missing = X_missing.fillna(X_alcohol.median())
    
    X_missing_scaled = scaler_alcohol.transform(X_missing)
    y_missing_pred = np.argmax(model_alcohol.predict(X_missing_scaled, verbose=0), axis=1)
    alcohol_freq_predicted = label_encoder.inverse_transform(y_missing_pred)
    
    # Actualizar
    df_completed = df.copy()
    missing_indices = df[df['alcohol_freq'].isna()].index
    df_completed.loc[missing_indices, 'alcohol_freq'] = alcohol_freq_predicted
    df = df_completed.copy()
    
    print(f"✓ {n_missing} valores predichos e integrados")
else:
    print("✓ No hay valores faltantes en alcohol_freq")

print("="*80)

## 4. Preprocesamiento Final
Excluir: person_id, annual_medical_cost, annual_premium, monthly_premium

In [ ]:
print("\n" + "="*80)
print("PREPROCESAMIENTO FINAL")
print("="*80)

target = 'annual_medical_cost'
exclude_cols = ['person_id', target, 'annual_premium', 'monthly_premium']

features = df.drop(columns=exclude_cols, errors='ignore')
categorical_features = features.select_dtypes(include=['category', 'object']).columns.tolist()
features_encoded = pd.get_dummies(features, columns=categorical_features, drop_first=True)
features_encoded = features_encoded.fillna(features_encoded.median())

X = features_encoded.values
y = df[target].values

print(f"\nFeatures: {X.shape[1]} columnas")
print(f"Excluidas: {exclude_cols}")
print(f"Samples: {len(X)}")
print(f"Target - Rango: [{y.min():.2f}, {y.max():.2f}], Media: {y.mean():.2f}")
print("="*80)

## 5. División Train/Test (70/30)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print(f"\nDatos de entrenamiento: {len(X_train)} (70%)")
print(f"Datos de test: {len(X_test)} (30%)")

# Normalizar
scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train)
X_test_scaled = scaler_final.transform(X_test)

print("\n✓ Datos normalizados")

## 6. Crear Modelo con Configuración Óptima

In [ ]:
print("\n" + "="*80)
print("CREANDO MODELO FINAL")
print("="*80)

# Construir modelo según configuración
final_model = models.Sequential()
final_model.add(layers.Input(shape=(X_train_scaled.shape[1],)))

# Capas ocultas
for i in range(best_config['n_layers']):
    final_model.add(layers.Dense(
        best_config['nodes_per_layer'][i], 
        activation=best_config['activation']
    ))
    
    if best_config['use_batch_norm']:
        final_model.add(layers.BatchNormalization(
            momentum=best_config['batch_norm_momentum']
        ))
    
    if best_config['dropout_rates'][i] > 0.01:
        final_model.add(layers.Dropout(best_config['dropout_rates'][i]))

# Capa de salida
final_model.add(layers.Dense(1, activation='linear'))

# Compilar
final_model.compile(
    optimizer=Adam(learning_rate=best_config['learning_rate']),
    loss='mse',
    metrics=['mae']
)

print("\nResumen del modelo:")
final_model.summary()

## 7. Entrenamiento Final

In [ ]:
print("\n" + "="*80)
print("ENTRENAMIENTO FINAL")
print("="*80)

# Callbacks
early_stop_final = callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)

reduce_lr_final = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1
)

# Entrenar
print("\nIniciando entrenamiento...")
train_start = time.time()

history_final = final_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop_final, reduce_lr_final],
    verbose=1
)

train_time = time.time() - train_start
print(f"\n✓ Entrenamiento completado en {train_time/60:.2f} minutos")

## 8. Evaluación Final

In [ ]:
print("\n" + "="*80)
print("EVALUACIÓN FINAL")
print("="*80)

# Predecir
y_pred_final = final_model.predict(X_test_scaled, verbose=0).flatten()

# Métricas
mae_final = mean_absolute_error(y_test, y_pred_final)
mse_final = mean_squared_error(y_test, y_pred_final)
rmse_final = np.sqrt(mse_final)
r2_final = r2_score(y_test, y_pred_final)

print("\nMÉTRICAS DEL MODELO FINAL:")
print(f"  MAE:  {mae_final:.4f}")
print(f"  MSE:  {mse_final:.4f}")
print(f"  RMSE: {rmse_final:.4f}")
print(f"  R²:   {r2_final:.4f}")

print("\nCOMPARACIÓN CON BASELINE:")
print(f"  MAE Baseline:  {best_config['mae_baseline']:.4f}")
print(f"  MAE Final:     {mae_final:.4f}")
improvement = (best_config['mae_baseline'] - mae_final) / best_config['mae_baseline'] * 100
print(f"  Mejora:        {improvement:.2f}%")

if mae_final < best_config['mae_baseline']:
    print("\n✓ OBJETIVO CUMPLIDO: El modelo supera al baseline")
else:
    print("\n✗ Advertencia: El modelo no supera al baseline")

print("="*80)

## 9. Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Loss
axes[0, 0].plot(history_final.history['loss'], label='Train Loss')
axes[0, 0].plot(history_final.history['val_loss'], label='Val Loss')
axes[0, 0].set_xlabel('Época')
axes[0, 0].set_ylabel('Loss (MSE)')
axes[0, 0].set_title('Evolución de la Pérdida')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# MAE
axes[0, 1].plot(history_final.history['mae'], label='Train MAE')
axes[0, 1].plot(history_final.history['val_mae'], label='Val MAE')
axes[0, 1].axhline(y=best_config['mae_baseline'], color='r', linestyle='--', label='Baseline')
axes[0, 1].set_xlabel('Época')
axes[0, 1].set_ylabel('MAE')
axes[0, 1].set_title('Evolución del MAE')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Predicciones vs Real
axes[1, 0].scatter(y_test, y_pred_final, alpha=0.5, s=10)
axes[1, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1, 0].set_xlabel('Costo Real')
axes[1, 0].set_ylabel('Costo Predicho')
axes[1, 0].set_title('Predicciones vs Valores Reales')
axes[1, 0].grid(True, alpha=0.3)

# Distribución de errores
errors = y_test - y_pred_final
axes[1, 1].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[1, 1].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1, 1].set_xlabel('Error (Real - Predicho)')
axes[1, 1].set_ylabel('Frecuencia')
axes[1, 1].set_title('Distribución de Errores')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('final_model_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico guardado: final_model_results.png")

## 10. Guardar Modelo

In [ ]:
# Guardar modelo
final_model.save('modelo_final_ff.h5')
print("\n✓ Modelo guardado: modelo_final_ff.h5")

# Guardar resultados
results = {
    'architecture': best_config,
    'training_time_minutes': train_time / 60,
    'metrics': {
        'mae': float(mae_final),
        'mse': float(mse_final),
        'rmse': float(rmse_final),
        'r2': float(r2_final)
    },
    'baseline_mae': best_config['mae_baseline'],
    'improvement_percentage': float(improvement),
    'training_samples': len(X_train),
    'test_samples': len(X_test)
}

with open('final_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print("✓ Resultados guardados: final_results.json")

## 11. Resumen de Tiempos

In [ ]:
total_time = time.time() - start_total

print("\n" + "="*80)
print("RESUMEN DE TIEMPOS")
print("="*80)
print(f"Tiempo total de ejecución: {total_time/60:.2f} minutos ({total_time/3600:.2f} horas)")
print(f"Tiempo de entrenamiento: {train_time/60:.2f} minutos")
print(f"\nFin: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

print("\n✓✓✓ TRABAJO PRÁCTICO COMPLETADO EXITOSAMENTE ✓✓✓")